In [ ]:
import sys
from PyQt5.QtWidgets import *
import os
import pandas as pd

class Popup(QWidget):
    def __init__(self):
        # Constructor for MyWindow
        super().__init__()
        
        # Construct user interface controller
        self.init_gui()

        # Initialize the user interface
    def init_gui(self):
        self.setGeometry(250, 250, 420, 225)
        self.setWindowTitle('Phone Case Generator')

        # Ask the user for wall thickness and clearance
        # Labels
        self.text1 = QLabel(self)
        self.text1.setText("Select the case's thickness (mm)")
        self.text1.move(20, 20)

        self.text2 = QLabel(self)
        self.text2.setText("Select the clearance between the case and the phone (mm)")
        self.text2.move(20, 80)

        # Input fields
        self.thickness = QLineEdit(self)
        self.thickness.resize(200, 25)
        self.thickness.move(20, 40)
        
        self.clearance = QLineEdit(self)
        self.clearance.resize(200, 25)
        self.clearance.move(20, 100)
        
        
        # Create the dropdown and label
        self.text1 = QLabel(self)
        self.text1.setText("Select your phone")
        self.text1.move(20, 140)

        self.dropdown = QComboBox(self)
        self.dropdown.move(20, 165)
        self.dropdown.resize(200, 30)

        # Adding items to the dropdown
        df = pd.read_csv("phone_dimensions.csv")     
        options = []
        for i in range (len(df.columns)):
            if i != 0:
                options.append(df.columns[i])
        self.dropdown.addItems(options)

        # Create the Generate button
        self.btn_generate = QPushButton("Generate code", self)
        self.btn_generate.resize(100, 30)
        self.btn_generate.move(310, 130)
        self.btn_generate.clicked.connect(self.click_gcode)

        # Create the Exit button
        self.btn_exit = QPushButton("Exit program", self)
        self.btn_exit.resize(100, 30)
        self.btn_exit.move(310, 165)
        self.btn_exit.clicked.connect(self.click_exit)
        
        # Show the window
        self.show()


        # Write the code
    def click_gcode(self):
        phone = self.dropdown.currentText()
        clearance = self.clearance.text()
        wall_thickness = self.thickness.text()
        back_thickness = wall_thickness

        # Define the variables according to the phone selected
        values = []
        df = pd.read_csv("phone_dimensions.csv")
        for i in range (len(df)):
            values.append(float(df.iloc[i, df.columns.get_loc(phone)]))
            
        # Generate the OpenSCAD code for the phone case
        scad_variables = f"""

// Auto-generated OpenSCAD code for {phone}

//----- Variables---- 

phone_height = {values[0]};
phone_width = {values[1]};
phone_depth = {values[2]};
phone_radius = {values[3]};
wall_thickness = {wall_thickness};
back_thickness = {back_thickness};
clearance = {clearance};
camera_width = {values[4]};
camera_height = {values[5]};
camera_radius = {values[6]};
camera_offset_y = {values[7]};
camera_offset_x = {values[8]};
power_side = {values[9]};
volume_side = {values[10]};
extra_side = {values[11]};
power_y = {values[12]};
power_width = {values[13]};
volume_y = {values[14]};
volume_width = {values[15]};
extra_y = {values[16]};
extra_width = {values[17]};
camera_button_y = {values[18]};
camera_button_width = {values[19]};
USB_width = {values[20]};
USB_height = {values[21]};
camera_side = {values[22]};
camera_button_height = {values[23]};
camera_button_radius = {values[24]};
lip_overlap = {values[25]};
lip_thickness = {values[26]};
fillet_radius = {values[27]};
button_radius = {values[28]};
button_height = {values[29]};

"""
        scad_body = """

//---- Modules ----

module rounded_rect(w, h, d, r) {
    hull() {
        translate([-w/2+r, -h/2+r, 0]) cylinder(r=r, h=d, $fn=64);
        translate([w/2-r,  -h/2+r, 0]) cylinder(r=r, h=d, $fn=64);
        translate([-w/2+r,  h/2-r, 0]) cylinder(r=r, h=d, $fn=64);
        translate([w/2-r,   h/2-r, 0]) cylinder(r=r, h=d, $fn=64);
    }
}

module phone_case() {
    case_w = phone_width + (clearance * 2) + (wall_thickness * 2);
    case_h = phone_height + (clearance * 2) + (wall_thickness * 2);
    case_d = phone_depth + clearance + back_thickness + lip_thickness; 
    case_r = phone_radius + wall_thickness;
    
    
    difference() {
        //----Case----
        translate([0, 0, fillet_radius])
            minkowski() {
                
                rounded_rect(
                    case_w - fillet_radius*2, 
                    case_h - fillet_radius*2, 
                    case_d - fillet_radius*2, 
                    max(0.1, case_r - fillet_radius));
                sphere(r=fillet_radius, $fn=24); 
            }
        //----Phone----
        translate([0, 0, back_thickness])
            rounded_rect(phone_width + clearance*2, phone_height + clearance*2, phone_depth + clearance, phone_radius);
        //----Lip----
        translate([0, 0, back_thickness + phone_depth + clearance +fillet_radius - 0.1])
            minkowski() {
                
                rounded_rect(
                    phone_width - (lip_overlap * 2) - (fillet_radius * 2), 
                    phone_height - (lip_overlap * 2) - (fillet_radius * 2), 
                    lip_thickness + 2 - (fillet_radius * 2), 
                    max(0.1, phone_radius - lip_overlap - fillet_radius));
                sphere(r=fillet_radius, $fn=24); 
            }
        //----Camera----
        translate([camera_offset_x, camera_offset_y, -1])
            rounded_rect(camera_width, camera_height, back_thickness + 2, camera_radius);
        //----USB-----
        translate([0, -case_h/2, phone_depth/2 - back_thickness])
            rotate([90, 0, 0]) 
            translate([0, wall_thickness*2, -wall_thickness*2]) 
            rounded_rect(USB_width, USB_height, wall_thickness*4, button_radius);
        //----Power----
        translate([case_w/2*power_side, power_y, phone_depth/2 - back_thickness])
            rotate([0, 90, 0]) 
            translate([-phone_depth/2+back_thickness/2, 0, -wall_thickness*2]) 
            rounded_rect(button_height, power_width, wall_thickness*4, button_radius);
        //----Volume----
        translate([case_w/2*volume_side, volume_y, phone_depth/2 - back_thickness])
            rotate([0, 90, 0]) 
            translate([-phone_depth/2+back_thickness/2, 0, -wall_thickness*2]) 
            rounded_rect(button_height, volume_width, wall_thickness*4, button_radius);
        //----Extra----
        translate([case_w/2*extra_side, extra_y, phone_depth/2 - back_thickness])
            rotate([0, 90, 0]) 
            translate([-phone_depth/2+back_thickness/2, 0, -wall_thickness*2]) 
            rounded_rect(button_height, extra_width, wall_thickness*4, button_radius);
        //----Camera button----
        translate([case_w/2*camera_side, camera_button_y, phone_depth/2 - back_thickness])
            rotate([0, 90, 0]) 
            translate([-phone_depth/2+back_thickness/2, 0, -wall_thickness*2]) 
            rounded_rect(camera_button_height, camera_button_width, wall_thickness*4, camera_button_radius);
    }
}


phone_case();
"""
        
        # Gets the path to the current user's home directory
        home = os.path.expanduser("~")
        file_path = os.path.join(home, "Documents", f"generated {phone} case.scad")
        with open(file_path, "w") as file:
                file.write(scad_variables + scad_body)

        # Inform the user of the success of the code printing
        QMessageBox.information(self, "Great success!", "The code was generated successfully\nand can be found in your 'documents' folder") 

    def click_exit(self):
        self.close()

if __name__ == "__main__":
    # Create an application object
    app = QApplication(sys.argv)
    
    # Create the user interface / Window
    window = Popup()
    
    # Create an event
    sys.exit(app.exec_())